<a href="https://colab.research.google.com/github/Mohamed-Shawky281/Hybrid-RAG-Research-assistant/blob/main/Hybrid_RAG_Research_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Hybrid RAG - Research Assistant Project

##Downloading libraries and dependencies...

In [1]:
!pip install -q langchain langchain-community langchain-chroma chromadb pypdf sentence-transformers rank_bm25
!pip install -q langchain-classic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 629.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/rag_research_assistant"
os.makedirs(f"{PROJECT_DIR}/papers", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/processed", exist_ok=True)
print("Project folder ready at:", PROJECT_DIR)

Mounted at /content/drive
Project folder ready at: /content/drive/MyDrive/rag_research_assistant


In [3]:
from pathlib import Path

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))
print(f"{len(pdf_paths)} PDF(s) found in papers/:")
for p in pdf_paths:
    print(" -", p.name)

31 PDF(s) found in papers/:
 - CryptographyinPostQuantumComputingEra.pdf
 - DOC-20260827-WA0014_260915_154811.pdf
 - DOC-20260822-WA0006_260915_154729.pdf
 - DOC-20260827-WA0016_260915_155026.pdf
 - DOC-20260827-WA0015_260915_155007.pdf
 - CryptographyinPostQuantumComputingEra (1).pdf
 - DOC-20260827-WA0014_260915_154811 (1).pdf
 - DOC-20260827-WA0016_260915_155026 (1).pdf
 - DOC-20260822-WA0006_260915_154729 (1).pdf
 - DOC-20260827-WA0015_260915_155007 (1).pdf
 - CryptographyinPostQuantumComputingEra (2).pdf
 - DOC-20260822-WA0006_260915_154729 (2).pdf
 - DOC-20260827-WA0014_260915_154811 (2).pdf
 - DOC-20260827-WA0015_260915_155007 (2).pdf
 - DOC-20260827-WA0016_260915_155026 (2).pdf
 - CryptographyinPostQuantumComputingEra (3).pdf
 - DOC-20260822-WA0006_260915_154729 (3).pdf
 - DOC-20260827-WA0014_260915_154811 (3).pdf
 - DOC-20260827-WA0015_260915_155007 (3).pdf
 - DOC-20260827-WA0016_260915_155026 (3).pdf
 - CryptographyinPostQuantumComputingEra (4).pdf
 - DOC-20260822-WA0006_2609

Uploading The wanted reference documents

In [4]:
from google.colab import files

uploaded = files.upload()
for fname in uploaded.keys():
    dest = f"{PROJECT_DIR}/papers/{fname}"
    with open(dest, "wb") as f:
        f.write(uploaded[fname])
    print(f"Saved {fname} -> {dest}")

Saving 2025.acl-long.440.pdf to 2025.acl-long.440.pdf
Saving CryptographyinPostQuantumComputingEra.pdf to CryptographyinPostQuantumComputingEra.pdf
Saving DOC-20260822-WA0006_260915_154729.pdf to DOC-20260822-WA0006_260915_154729.pdf
Saving DOC-20260827-WA0014_260915_154811.pdf to DOC-20260827-WA0014_260915_154811.pdf
Saving DOC-20260827-WA0015_260915_155007.pdf to DOC-20260827-WA0015_260915_155007.pdf
Saving DOC-20260827-WA0016_260915_155026.pdf to DOC-20260827-WA0016_260915_155026.pdf
Saved 2025.acl-long.440.pdf -> /content/drive/MyDrive/rag_research_assistant/papers/2025.acl-long.440.pdf
Saved CryptographyinPostQuantumComputingEra.pdf -> /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf
Saved DOC-20260822-WA0006_260915_154729.pdf -> /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260822-WA0006_260915_154729.pdf
Saved DOC-20260827-WA0014_260915_154811.pdf -> /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260827-WA001

##Loading PDFs into LangChain Documents

In [5]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))

all_docs = []
for path in pdf_paths:
    loader = PyPDFLoader(str(path))
    docs = loader.load()  # one Document per page, metadata already includes 'source' and 'page'
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} pages from {len(pdf_paths)} papers")
print(all_docs[0].page_content[:500])
print(all_docs[0].metadata)

/tmp/ipykernel_688/455741346.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 827 pages from 31 papers
1 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
 
 
 
Cryptography in Post Quantum Computing Era 
Neerav Sood 
Independent Researcher
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-24T13:31:21-05:00', 'author': 'TR', 'moddate': '2024-01-24T13:31:21-05:00', 'source': '/content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf', 'total_pages': 96, 'page': 0, 'page_label': '1'}


In [6]:
#Take into consideration like truncated output , so strip it to avoid retreivial of gaps or empty spaces
import re

def clean_text(text: str) -> str:
    """
    Cleans up common PDF-extraction artifacts before chunking:
    - collapses multiple blank lines/newlines into one
    - collapses runs of spaces/tabs into a single space
    - strips leading/trailing whitespace per line
    - joins words that got hyphen-broken across a line (common in PDFs)
    """
    # Fix hyphenated words split across a line break, e.g. "crypto-\ngraphy" -> "cryptography"
    text = re.sub(r"-\n", "", text)

    # Collapse 3+ newlines (big gaps) down to a single paragraph break
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Collapse runs of spaces/tabs into one space
    text = re.sub(r"[ \t]+", " ", text)

    # Strip trailing whitespace on each line
    text = "\n".join(line.strip() for line in text.split("\n"))

    return text.strip()


for doc in all_docs:
    doc.page_content = clean_text(doc.page_content)

print("Cleaned text for all", len(all_docs), "pages")
print(all_docs[0].page_content[:500])  # sanity check -- compare to before cleaning

Cleaned text for all 827 pages
1
CRYPTOGRAPHY IN POST-QUANTUM ERA





Cryptography in Post Quantum Computing Era
Neerav Sood
Independent Researcher


##Chunking (including overlapping) & tuned

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,      # characters -- LangChain's default unit, Also to be tuned
    chunk_overlap=500,
    separators=["\n\n", "\n", ". ", " ", ""],  # tries paragraph breaks first, falls back to smaller units
)

split_docs = splitter.split_documents(all_docs)
print(f"Created {len(split_docs)} chunks from {len(all_docs)} pages")
print(split_docs[0].page_content[:300])
print(split_docs[0].metadata)

Created 2520 chunks from 827 pages
1
CRYPTOGRAPHY IN POST-QUANTUM ERA





Cryptography in Post Quantum Computing Era
Neerav Sood
Independent Researcher
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-24T13:31:21-05:00', 'author': 'TR', 'moddate': '2024-01-24T13:31:21-05:00', 'source': '/content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf', 'total_pages': 96, 'page': 0, 'page_label': '1'}


##Building the Chroma vector store (Meaning Similarity)

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=f"{PROJECT_DIR}/chroma_db",
)

print(f"Chroma vector store built with {vectorstore._collection.count()} chunks")

/tmp/ipykernel_688/2980357329.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma vector store built with 15294 chunks


In [9]:
print(f"Chroma vector store built with {vectorstore._collection.count()} chunks")

Chroma vector store built with 15294 chunks


In [10]:
import pickle
import os

file_path = f"{PROJECT_DIR}/processed/split_docs.pkl"

# Check if the file exists. If not, and split_docs is available, save it.
# This assumes split_docs is defined in the current kernel state from previous cells.
if not os.path.exists(file_path):
    print(f"File '{file_path}' not found. Saving 'split_docs' to file first.")
    with open(file_path, "wb") as f:
        pickle.dump(split_docs, f)

# Now, attempt to load the file (which should now exist or existed already)
with open(file_path, "rb") as f:
    split_docs = pickle.load(f)

print(f"Loaded {len(split_docs)} chunks")

Loaded 920 chunks


##Build the Keyword similarity (BM25 retriever)

In [11]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 5  # how many chunks BM25 returns per query

print("BM25 retriever built")

BM25 retriever built


In [12]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector retriever built")

Vector retriever built


##Merging of Both methods and tuning

In [13]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.65, 0.35], # The ratio of relavence used between vector and bm-25
)

print("Ensemble retriever built")

Ensemble retriever built


In [14]:
#Testing of retriver and accurate retrivial on a paper
query = "What wrong with Traditional cryptographic systems compared to quantum ones"

results = ensemble_retriever.invoke(query)

print(f"Returned {len(results)} chunks\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(doc.page_content[:300])
    print()

Returned 5 chunks

--- Result 1 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra (1).pdf | Page: 19
20 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
The robustness of lattice-based cryptographic systems against both classical and quantum 
attacks, coupled with their adaptability to a wide range of cryptographic applications, makes 
them a cornerstone in the quest for secure communication and data protect

--- Result 2 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf | Page: 67
68 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
 Another critical area of research is the practical integration of Post-Quantum 
Cryptography (PQC) algorithms into existing digital infrastructures. This requires understanding 
current systems' limitations and developing strategies for integrating new algo

--- Result 3 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputin

In [15]:
#Testing of retriver and accurate retrivial on a paper
query = "What problems are faced in multi-resource LLMs? "

results = ensemble_retriever.invoke(query)

print(f"Returned {len(results)} chunks\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(doc.page_content[:300])
    print()

Returned 4 chunks

--- Result 1 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260827-WA0014_260915_154811 (1).pdf | Page: 3
necessitating extensive adjustments to other components.
1) Planning: First of all, to incorporate the cases which
does not require any sources of external knowledge, we define
several additional indicate tokens, corresponding to different
sources, including the NULL token which signifies that there

--- Result 2 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf | Page: 14
complexity, making these problems intractable for both classical and quantum computers. 
The reason why lattice-based cryptographic methods are resistant to quantum computing 
attacks lies in the nature of these lattice problems. Unlike problems such as integer factorization 
or discrete logarithms,

--- Result 3 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260827-WA0016_260915_155026 (1).

##Setting API env for Generation

In [16]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.1 MB/s eta 0:00:00


In [17]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

print("Groq client ready")

Groq client ready


In [18]:
import requests

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {api_key}"}
)
models = resp.json()

for m in models["data"]:
    print(m["id"])

groq/compound-mini
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
groq/compound
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b


In [19]:
#Testing of API
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)
print(response.choices[0].message.content)

Hello!


In [20]:
def build_context(docs):
    """Formats retrieved chunks into a numbered context block with citations."""
    context_parts = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown").split("/")[-1]  # just filename, not full path
        page = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
        context_parts.append(f"[{i+1}] (Source: {source}, Page: {page})\n{doc.page_content}")
    return "\n\n".join(context_parts)


def build_prompt(query, docs):
    context = build_context(docs)
    prompt = f"""You are a research assistant answering questions based ONLY on the provided paper excerpts below.

Rules:
- Answer using ONLY information found in the excerpts below.
- If the excerpts don't contain enough information to answer, say so clearly -- do not use outside knowledge.
- Cite your sources using the [number] format matching the excerpts (e.g. "Quantum computing threatens RSA encryption [1].").
- Be concise and direct.

Excerpts:
{context}

Question: {query}

Answer:"""
    return prompt

In [21]:
def ask(query, retriever=ensemble_retriever, llm_model="openai/gpt-oss-120b"):
    docs = retriever.invoke(query)
    prompt = build_prompt(query, docs)

    response = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content
    return answer, docs


# Test it
query = "What is SouLLMate? and what is new value it provides?"
answer, source_docs = ask(query)

print("ANSWER:\n")
print(answer)
print("\n\nSOURCES USED:")
for i, doc in enumerate(source_docs):
    source = doc.metadata.get("source", "unknown").split("/")[-1]
    page = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
    print(f"[{i+1}] {source}, page {page}")

ANSWER:

The provided excerpts do not contain any information about “SouLLMate” or the new value it provides. Therefore, I cannot answer the question based on the given material.


SOURCES USED:
[1] DOC-20260827-WA0016_260915_155026.pdf, page 2
[2] CryptographyinPostQuantumComputingEra.pdf, page 17
[3] CryptographyinPostQuantumComputingEra (1).pdf, page 17
[4] CryptographyinPostQuantumComputingEra.pdf, page 28


In [22]:
!pip install -q gradio

In [23]:
import gradio as gr

custom_css = """
#main-title { font-size: 2.2em; font-weight: 700; color: #1a1a2e; }
.gradio-container { font-family: 'Inter', sans-serif; background: #f7f7fb; }
#answer-box textarea { font-size: 1.05em; line-height: 1.6; }
"""

def rag_interface(query):
    if not query.strip():
        return "Please enter a question.", ""

    answer, docs = ask(query)

    sources_text = ""
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown").split("/")[-1]
        page = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
        sources_text += f"**[{i+1}]** {source}, page {page}\n\n"

    return answer, sources_text


with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="Research Assistant") as demo:
    gr.Markdown(
        """
        # 📚 Hybrid RAG Research Assistant
        Ask questions about your uploaded research papers. Answers are grounded in retrieved excerpts, with citations you can verify.
        """,
        elem_id="main-title"
    )

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(
                label="Your question",
                placeholder="e.g. What post-quantum algorithms are discussed?",
                lines=2,
            )
            submit_btn = gr.Button("Ask", variant="primary")

            gr.Examples(
                examples=[
                    "What post-quantum cryptographic algorithms are discussed?",
                    "What is the main contribution of these papers?",
                    "What evaluation methods were used?",
                ],
                inputs=query_input,
            )

        with gr.Column(scale=3):
            answer_output = gr.Textbox(label="Answer", lines=10, elem_id="answer-box")
            sources_output = gr.Markdown(label="Sources")

    submit_btn.click(fn=rag_interface, inputs=query_input, outputs=[answer_output, sources_output])
    query_input.submit(fn=rag_interface, inputs=query_input, outputs=[answer_output, sources_output])

demo.launch(share=True)

/tmp/ipykernel_688/1675770299.py:24: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="Research Assistant") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ebd5b8470001796b52.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
